# Notebook 3: Sentinel-1 time-series analysis


This Notebook will take you through the steps of analysing persistent scatterers, taking single difference, double difference and building arcs.

## Preparations

1. Download this notebook and put it in its own directory
2. Download the data `s1_asc_t008_v2.nc` by using this link https://surfdrive.surf.nl/s/HtX5nCEQoJrnyrH 

(copy the path of the datafile)

Activate environment:

`conda activate workshop-env` (or your own environment)



In [ ]:
import os
import numpy as np

import xarray as xr
import matplotlib.pyplot as plt



from tqdm import tqdm
from datetime import datetime

from drama import utils

import contextily as cx


# Load the data by building a path.

The data is from sentinel 1 the ascending 088 track

In [ ]:
track = 's1_asc_t088'
path = os.path.join('s1_asc_t088_v2.nc', track) 
netcdf_file = os.path.join('/Users/isabelslingerland/insar-1/book/', ("%s_v2.nc" % track)) # path to the netcdf file

s1_stack = xr.load_dataset(netcdf_file,engine="h5netcdf") #load the netcdf file as an xarray dataset


#inspect xarray dataset
s1_stack

In [ ]:

s1_complex = s1_stack.complex.values #select the complex values from the xarray dataset
lat = s1_stack.lat.values #select the latitude values from the xarray dataset
lon = s1_stack.lon.values #select the longitude values from the xarray dataset

## Make a spatial plot to visualize the data at the first epoch

<div style="background-color:#AABAB2; color: black; vertical-align: middle; padding:15px; margin: 10px; border-radius: 10px">
<p>
<b>Task 1.1: </b>   
    
Make a plot to visualise the amplitude in the stack:
- visualise an amplitude image for the first time step (amp0)
- and an average amplitude image over the whole stack(amp_tmpavg). 

</p>
</div>

In [ ]:
#compute the amplitude for the first time step 
amp0 = #YOUR CODE HERE
mean0 = np.nanmean(amp0) # compute the mean amplitude for the first time step, ignoring NaN values
std0 = np.std(amp0) #compute the standard deviation of the amplitude for the first time step

#compute the average amplitude across all time steps
amp_tmpavg = #YOUR CODE HERE

#lets plot
plt.figure(figsize=(12,6))
plt.subplot(1,2,1)
plt.imshow(amp0, origin='lower', aspect=4, vmax=(mean0+2*std0), cmap='inferno')
plt.colorbar()
plt.title('Amplitude at first time step')
plt.subplot(1,2,2)
plt.imshow(amp_tmpavg, origin='lower', aspect=4, vmax=(mean0+1*std0), cmap='inferno')
plt.colorbar()
plt.title('Average Amplitude')
plt.show()



The scene is in the North of the Netherlands where there is ground displacement due to gas extraction. The urban area you see is a small town called Roden.

In [ ]:
lon_sorted = np.sort(lon)
lat_sorted = np.sort(lat)

plt.figure(figsize=(14,6))

ax1 = plt.subplot(1, 2, 1)
plt.pcolormesh(lon_sorted, lat_sorted, amp0, shading='auto', cmap='inferno', vmax=(mean0 + 2 * std0), alpha = 0.85)
plt.colorbar(fraction = 0.03)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
cx.add_basemap(ax1, crs='epsg:4326', source=cx.providers.OpenStreetMap.Mapnik, zoom = 15)


# Tweede subplot
ax2 = plt.subplot(1, 2, 2)
plt.pcolormesh(lon_sorted, lat_sorted, amp_tmpavg, shading='auto', cmap='inferno', vmax=(mean0 + 1 * std0), alpha = 0.85)
plt.colorbar(fraction = 0.03)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
cx.add_basemap(ax2, crs='epsg:4326', source=cx.providers.OpenStreetMap.Mapnik, zoom = 15)
plt.tight_layout()

plt.figure(figsize=(14,6))

ax1 = plt.subplot(1, 2, 1)
plt.pcolormesh(lon_sorted, lat_sorted, amp0, shading='auto', cmap='inferno', vmax=(mean0 + 2 * std0), alpha = 0.01)
plt.colorbar(fraction = 0.03)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
cx.add_basemap(ax1, crs='epsg:4326', source=cx.providers.OpenStreetMap.Mapnik, zoom = 15)


# Tweede subplot
ax2 = plt.subplot(1, 2, 2)
plt.pcolormesh(lon_sorted, lat_sorted, amp_tmpavg, shading='auto', cmap='inferno', vmax=(mean0 + 1 * std0), alpha = 0.01)
plt.colorbar(fraction = 0.03)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
cx.add_basemap(ax2, crs='epsg:4326', source=cx.providers.OpenStreetMap.Mapnik, zoom = 15)

plt.tight_layout()
plt.show()



## Amplitude calibration
We should check that the data is calibrated in amplitude, or trust that it is. One way is to check the mean intensity at different epochs.

<div style="background-color:#AABAB2; color: black; vertical-align: middle; padding:15px; margin: 10px; border-radius: 10px">
<p>
<b>Task 1.2: </b>   
    
Calculate and plot the mean and median amplitude over time in dB:
$$
   A_{dB,i} = 20 \cdot log_{10}\left(\frac{<A>_i}{max_i(<A>_i)}\right)
$$




Plot both the mean and median on the same figure.  

**Is the amplitude stable over time?**

</p>
</div>



In [ ]:

A_all = #YOUR CODE HERE
mean_A = #YOUR CODE HERE
max_mean_A = #YOUR CODE HERE
median_A = #YOUR CODE HERE
max_median_A = #YOUR CODE HERE

# Make your plot here

<div style="background-color:#AABAB2; color: black; vertical-align: middle; padding:15px; margin: 10px; border-radius: 10px">
<p>
<b>Task 1.3: Amplitude dispersion</b>   


If the amplitude is not stable over time, the data should be calibrated first. 
One way is to divide each image by the mean amplitude:

$$SLC_{i,cal} = \frac{SLC_i}{\sqrt{\langle A \rangle_i}}$$

Compute and plot the amplitude dispersion $D_a$, with and without this calibration:

$$D_a = \frac{\sigma_a}{m_a}$$

where $\sigma_a$ and $m_a$ are the standard deviation and mean of the amplitude of each pixel over time.



</p>
</div>



In [ ]:
# Compute the amplitude dispersion
Da = #YOUR CODE HERE

#try to calibrate the amplitude by dividing by the mean amplitude
amp_cal = #YOUR CODE HERE
#then compute the amplitude dispersion of the calibrated amplitude
Da_cal = #YOUR CODE HERE


In [ ]:
plt.figure(figsize=(12,6))
plt.subplot(1,2,1)
plt.imshow(Da, origin='lower', aspect=4, cmap='inferno_r',vmin=0.2, vmax=0.6)
plt.colorbar()
#plt.figure(figsize=(8,8))
plt.subplot(1,2,2)
plt.imshow(Da_cal, origin='lower', aspect=4, cmap='inferno_r', vmin=0.2, vmax=0.6)
plt.colorbar()


<div style="background-color:#AABAB2; color: black; vertical-align: middle; padding:15px; margin: 10px; border-radius: 10px">
<p>
<b>Task 1.4: Selecting PS</b>   

Select potential Persistent Scatterers (PS) by making a threshold for the the amplitude dispersion.

You could use `np.where` to find the pixel positions where $D_a$ is below a threshold of 0.25:

Do this for both the calibrated and uncalibrated amplitude dispersion.

- How many PS candidates are selected in each case?
- Does the calibration make a difference?
- plot the potential PS positions on a map
- You could experiment with the threshold




</p>
</div>



In [ ]:
#code here to select pixels with low amplitude dispersion

ps_pos = #YOUR CODE HERE
ps_cal_pos = #YOUR CODE HERE


#let's select the lat and lon coordinates (this was in your data) of the pixels with low amplitude dispersion
ps_lat = #YOUR CODE HERE
ps_lon = #YOUR CODE HERE




Lets plot those selected points so we can see where they are on the map

In [ ]:
fig, ax = plt.subplots(figsize = (15,15))
plt.scatter(ps_lon, ps_lat, s=20, color = 'tab:red', label = 'PS')
#plt.scatter(point1[0], point1[1], s=50, color = 'tab:blue', label = 'point1')
plt.legend()
plt.ylim(np.min(lat), np.max(lat))
plt.xlim(np.min(lon), np.max(lon))
cx.add_basemap(ax, crs='epsg:4326', source=cx.providers.OpenStreetMap.Mapnik, zoom = 15)

## Part 2: Plot phase time series and Amplitude of some detected points

In [ ]:
#lets just set up our points for the next steps

points = np.array(ps_cal_pos).T #pnts idx
print('points.shape:', points.shape)
print('first 5 points:\n', points[:5,:])

# complex values time-series for selected points
slc_points = s1_stack.complex.values[points[:, 0], points[:, 1]] #shape is (n_points, n_time_steps))
#slc_points = stack[points[:, 0], points[:, 1]]
Da_points = Da_cal[points[:, 0], points[:, 1]] #shape is (n_points)

#lets separate the amplitude and phase from the complex values
ps_amplitude = np.abs(slc_points)
ps_phase = np.angle(slc_points)

#lets also get the dates for the time-series
dates = s1_stack.time.values



<div style="background-color:#AABAB2; color: black; vertical-align: middle; padding:15px; margin: 10px; border-radius: 10px">
<p>
<b>Task 2.1: Visualise the phase of a PS over time</b>   

pick a PS location and visualise:
- phase timeseries (you can add + & - 2 $\pi$ to visualise phase ambiguity)
- amplitude timeseries




</p>
</div>



In [ ]:
pnt_idx = #pick PS scatterer


#plot the phase timeseries for the point under consideration
#also plot the phase timeseries shifted by 2*pi and -2*pi to show the ambiguity of the phase




## Here we will visualise all PS positions on a map (those below the threshold) and our choosen point

In [ ]:
fig, ax = plt.subplots(figsize = (15,15))
plt.scatter(ps_lon, ps_lat, s=20, color = 'tab:red', label = 'PS')
plt.scatter(ps_lon[pnt_idx], ps_lat[pnt_idx], s=200, color = 'tab:blue', label = 'Point under consideration')
plt.legend()
plt.ylim(np.min(lat), np.max(lat))
plt.xlim(np.min(lon), np.max(lon))
cx.add_basemap(ax, crs='epsg:4326', source=cx.providers.OpenStreetMap.Mapnik, zoom = 15)

# Creating the temporal phase differences


<div style="background-color:#AABAB2; color: black; vertical-align: middle; padding:15px; margin: 10px; border-radius: 10px">
<p>
<b>Task 2.2: Choose a datum </b>   

We will choose a mother date thats considered our datum and we have to "subtract" the mother phase from the other dates.
We refer to this as **single differencing (SD)**


- pick a index point in from the slc points, this will be your mother date, and then take the difference in phase between the mother and all other slc points dates (differencing in time referring all dates to that mother date)


Remember how to do this with complex numbers 
$$ I_{12} = y_1 \cdot y_2^*$$

- once you have your single mother stack plot again the phase time series


__To start you can just randomly pick a date and see what happens, but its good to think about what are the effects of choosing a specific mother date__

- think about noise that might affect this reference date, consider temporal coherence
- what do you expect the phase at the mother epoch to be?
- How do you expect the phase timeseries of a persistent scatterer thats been single differenced to change? 
You can make a plot again of the same PS phase that you choose before and see how it changed.


 




</p>
</div>



In [ ]:
# Define mother epoch and compute single-mother stack 

idx_mother_date = #pick mother epoch
print (f'The reference date is {dates[idx_mother_date]}')

#subtract the complex values at the mother epoch from the complex values at all epochs to get the single-mother stack
ps_sd_points = #YOUR CODE HERE
ps_sd_phase = #YOUR CODE HERE

# test for the reference phase at the mother epoch
phase_at_mother_epoch = #YOUR CODE HERE




# Make your plot here to check the phase at the mother epoch is zero for all points. 
# Does this make sense? is it zero?

#plot the single-difference phase timeseries (ps_sd_phase) for the point under consideration (pnt_idx)



#plot the original SLC ps phase timeseries for the same point for comparison

#also plot the phase timeseries shifted by 2*pi and -2*pi to show the ambiguity of the phase for the single-difference stack





<div style="background-color:#AABAB2; color: black; vertical-align: middle; padding:15px; margin: 10px; border-radius: 10px">
<p>
<b>Task 2.3: Create Double Difference phase</b>   

We will need a reference point, another PS, to take the Double Difference of the phase of the point that we were considering. 
We are looking at the difference in phase between two point, we call this an *arc*




__Choose a reference point. We like our reference point to be a good stable PS__

- You can use amplitude dispersion (Da) to get a good point? A good point has a low amplitude dispersion
- It maybe good to check for a couple of low amplitude dispersion and look at the phase DD timeseries between two PS points. 
- Also plot the amplitude timeseries for both points the referece point (ref_pnt_idx) and the point we considered (pnt_idx). Always good to check.

</p>
</div>



In [ ]:
# We need a reference phase
# Best point, according to Da, take it as reference.

ref_pnt_idx = #YOUR CODE HERE

print("Reference position= %i, %i" % (points[ref_pnt_idx, 0], points[ref_pnt_idx, 1]))


dd_points = #YOUR CODE HERE
ps_dd_phase = #YOUR CODE HERE

#plot the DD phase timeseries for the point under consideration 
# again plot the phase timeseries shifted by 2*pi and -2*pi to show the ambiguity of the phase for the double-difference stack


#plot the amplitude of the point under consideration pnt_idx and for reference point ref_pnt_idx for comparison



In [ ]:
# visualise your points and arc on the map
fig, ax = plt.subplots(figsize = (15,15))
plt.scatter(ps_lon[ref_pnt_idx], ps_lat[ref_pnt_idx], s=200, color = 'tab:red', label = 'Reference point')
plt.scatter(ps_lon[pnt_idx], ps_lat[pnt_idx], s=200, color = 'tab:blue', label = 'Point under consideration')
plt.plot((ps_lon[ref_pnt_idx], ps_lon[pnt_idx]), (ps_lat[ref_pnt_idx], ps_lat[pnt_idx]), color = 'tab:gray', label = 'Arc between points')
plt.legend()
plt.ylim(np.min(lat), np.max(lat))
plt.xlim(np.min(lon), np.max(lon))
cx.add_basemap(ax, crs='epsg:4326', source=cx.providers.OpenStreetMap.Mapnik, zoom = 15)

In [ ]:
# You can also zoom into the area where is your reference point is it a building? a road? a field?

fig, ax = plt.subplots(figsize = (15,15))
plt.scatter(ps_lon[ref_pnt_idx], ps_lat[ref_pnt_idx], s=200, color = 'tab:red', label = 'Reference point')
plt.legend()
plt.xlim(ps_lon[ref_pnt_idx] - 0.003, ps_lon[ref_pnt_idx] + 0.003)
plt.ylim(ps_lat[ref_pnt_idx] - 0.003, ps_lat[ref_pnt_idx] + 0.003)
cx.add_basemap(ax, crs='epsg:4326', source=cx.providers.OpenStreetMap.Mapnik, zoom = 17)


<div style="background-color:#AABAB2; color: black; vertical-align: middle; padding:15px; margin: 10px; border-radius: 10px">
<p>
<b>Task 2.4: Check out different point and build arcs</b>   


It could be very good to inspect some different arcs, by choosing different points. 

- You can use the code above and check individual arcs.

- We can also just make an array of different points to consider and just check out some combinations.

- Lastly what happens if you choose a different reference point.

</p>
</div>



In [ ]:
sorted_ref_pnt_idx = np.argsort(Da_points)
print("Top 5 points with lowest Da:\n", sorted_ref_pnt_idx[:5])

In [ ]:

#make random selection of 5 points to plot the timeseries for
pnt_idx_range = np.random.choice(points.shape[0], size=5, replace=False) #randomly select 10 points from the points array


ref_pnt_idx = sorted_ref_pnt_idx[1] #you can try out the top 5 points with lowest Da as reference point and see how the timeseries look like for different points
                                 

print("Reference position= %i, %i" % (points[ref_pnt_idx, 0], points[ref_pnt_idx, 1]))

dd_points = sd_points * np.conj(sd_points[ref_pnt_idx, :][np.newaxis,:])
ps_dd_phase = np.angle(dd_points)

for i in range(pnt_idx_range.shape[0]):

    pnt_idx = pnt_idx_range[i]
    
    fig, (ax0, ax1, ax2) = plt.subplots(3, 1, figsize=(11, 11), sharex=True)

    # 1) Double-difference phase
    ax0.plot(dates, ps_dd_phase[pnt_idx, :], ".", label="DD phase", color="tab:orange")
    ax0.plot(dates, ps_dd_phase[pnt_idx, :] + 2*np.pi, ".", color="gray", label="+2π / -2π ambiguity")
    ax0.plot(dates, ps_dd_phase[pnt_idx, :] - 2*np.pi, ".", color="gray")
    ax0.set_ylabel("Phase (radians)")
    ax0.set_title(f"Double-difference phase time-series for point {pnt_idx}")
    ax0.legend()

    # 2) Amplitude for point under consideration
    ax1.plot(dates, 20*np.log10(ps_amplitude[pnt_idx, :]), label="Amplitude point under consideration")
    ax1.set_ylabel("Amplitude (dB)")
    ax1.set_title(f"Amplitude time-series for point {pnt_idx}")
    ax1.legend()

    # 3) Amplitude for reference point
    ax2.plot(dates, 20*np.log10(ps_amplitude[ref_pnt_idx, :]), label="Amplitude reference point", color="tab:orange")
    ax2.set_xlabel("Date")
    ax2.set_ylabel("Amplitude (dB)")
    ax2.set_title(f"Amplitude time-series for reference point {ref_pnt_idx}")
    ax2.legend()

    plt.tight_layout()
    plt.show()
    fig, ax = plt.subplots(figsize = (15,15))
    plt.scatter(ps_lon[ref_pnt_idx], ps_lat[ref_pnt_idx], s=200, color = 'tab:red', label = 'Reference point')
    plt.scatter(ps_lon[pnt_idx], ps_lat[pnt_idx], s=200, color = 'tab:blue', label = 'Point under consideration')
    plt.plot((ps_lon[ref_pnt_idx], ps_lon[pnt_idx]), (ps_lat[ref_pnt_idx], ps_lat[pnt_idx]), color = 'tab:gray', label = 'Arc between points')
    plt.legend()
    plt.ylim(np.min(lat), np.max(lat))
    plt.xlim(np.min(lon), np.max(lon))
    cx.add_basemap(ax, crs='epsg:4326', source=cx.providers.OpenStreetMap.Mapnik, zoom = 15)